# Solutions · 17 · Project: from a month of column data to a retrofit proposal

Worked solutions to the exercises of [notebook 17](../notebooks/17_project_column_data_to_retrofit.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engsep import datasets, vle
from engsep import distillation as dist
from engsep.vle import y_from_alpha
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
p = pd.read_csv(datasets.path("plant_column_benzene_toluene.csv"), comment="#")
q = p[p.note != "target for the retrofit"].copy()
q.loc[q.xD > 1, "xD"] /= 100; q.loc[q.T_top_C > 200, "T_top_C"] -= 273.15; q = q[q.zF > 0]
alpha = vle.geometric_mean_alpha(("benzene", "toluene"), ideal=True)
y_eq = lambda x: y_from_alpha(x, alpha)

## Exercise 1

The plant also asks whether the *feed tray* is in the right place. For the current operation, find the optimum feed stage from McCabe-Thiele and compare with the actual feed tray (tray 9 from the top). How many stages are lost if the feed is two trays too high?

In [3]:
row = q.iloc[0]
mt = dist.mccabe_thiele(y_eq, row.xD, row.xB, row.zF, row.R)
E = 10 / 18
opt_tray = round(mt["feed_stage"] / E)
print(f"day {int(row.day)}: optimum feed on theoretical stage {mt['feed_stage']} of {mt['stages']} -> real tray {opt_tray} from the top (actual feed tray 9)")
# stages lost with the feed two trays too high: step with the feed forced onto stage feed-2 (rectifying line continued past the intersection)
def stages_with_feed_at(k_feed):
    ol = mt["lines"]
    x = y = row.xD; n = 0
    while x > row.xB and n < 200:
        x = float(__import__("scipy").optimize.brentq(lambda xx: y_eq(xx) - y, 0, 1)); n += 1
        if x <= row.xB:
            break
        y = ol["rect"](x) if n < k_feed else ol["strip"](x)
        if y <= y_eq(x) and False:
            pass
    return n
print(f"stages needed with the feed at stage {mt['feed_stage'] - 2} instead of {mt['feed_stage']}: {stages_with_feed_at(mt['feed_stage'] - 2)} (optimal: {mt['stages']})")

day 1: optimum feed on theoretical stage 6 of 11 -> real tray 11 from the top (actual feed tray 9)
stages needed with the feed at stage 4 instead of 6: 13 (optimal: 11)


The optimum feed stage sits where the staircase crosses the operating lines' intersection - theoretical
stage 6, real tray 11 of 18 - two trays below the actual feed tray 9: the feed is entering slightly too high.
Forcing the feed two theoretical stages early makes the column step along the stripping line where the
rectifying line would have been better and costs two stages; moving a feed nozzle is a cheap fix for a
common inefficiency.

## Exercise 2

Hydraulics: at the retrofit reflux ratio the vapour rate rises. Using the flooding estimate of notebook 05 (vapour density 2.8 kg/m3, liquid 800, surface tension 0.021 N/m) and a 1.2 m shell, check that the packing operates below 80 % of flooding. If not, what reflux ratio is the hydraulic limit, and how many stages does that require?

In [4]:
rho_L, rho_V, sigma, D_shell = 800.0, 2.8, 0.021, 1.2
u_f = 0.08 * (sigma / 0.02) ** 0.2 * np.sqrt((rho_L - rho_V) / rho_V)
area = np.pi * D_shell ** 2 / 4
D_flow = 45.0 / 3600 * 29.0                                             # distillate molar flow x molar volume, m3/s per unit (R+1)
for R in (1.88, 2.5, 3.0):
    V = (R + 1) * D_flow
    u = V / area
    print(f"R = {R}: vapour velocity {u:.2f} m/s = {u/u_f:.0%} of flooding ({u_f:.2f} m/s)")
R_limit = 0.8 * u_f * area / D_flow - 1
print(f"80 % of flooding is reached at R = {R_limit:.2f}")

R = 1.88: vapour velocity 0.92 m/s = 68% of flooding (1.36 m/s)
R = 2.5: vapour velocity 1.12 m/s = 82% of flooding (1.36 m/s)
R = 3.0: vapour velocity 1.28 m/s = 94% of flooding (1.36 m/s)
80 % of flooding is reached at R = 2.40


The 1.2 m shell carries the retrofit reflux ratio of 1.9 at about two thirds of flooding, and the 80 %
limit is reached at R = 2.4: enough margin for normal operation, not for a large reflux increase. Structured
packing's higher capacity than trays at the same diameter (the Souders-Brown factor above is a tray value)
is one more argument for it, and the vendor's capacity curve should confirm the margin.

## Exercise 3

**Implement it yourself:** write `column_stages(df)` that returns the theoretical stage count for every day and its median, and confirm it reproduces the notebook's table.

In [5]:
def column_stages(df):
    n = df.apply(lambda r: dist.mccabe_thiele(y_eq, r.xD, r.xB, r.zF, r.R)["stages"], axis=1)
    return n, int(round(n.median()))
n, med = column_stages(q)
print("stage counts by day:", n.tolist()); print("median:", med)

stage counts by day: [11, 11, 11, 12, 11, 11, 11, 12, 12, 12, 12, 11, 12, 11, 12, 11, 12, 12, 11, 11, 12, 12, 11, 12, 11, 12, 11, 12, 11]
median: 11


One line per day and a median: the reproducible core of the project's performance assessment.